## Setup

In [ ]:
%config InlineBackend.figure_format = "retina"
from cycler import cycler
from matplotlib import colormaps, rcParams
from pypalettes import load_cmap

_cmap = load_cmap("default_jama")
if _cmap.name not in colormaps:
    colormaps.register(_cmap, name=_cmap.name)

rcParams["savefig.dpi"] = 100
rcParams["figure.dpi"] = 100
rcParams["font.size"] = 14

# Use LaTeX for all text rendering
rcParams["text.usetex"] = True
rcParams["font.family"] = "serif"
rcParams["font.serif"] = ["Computer Modern"]

# Ticks on all sides, pointing inward
rcParams["xtick.direction"] = "in"
rcParams["ytick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["ytick.right"] = True

# Minor ticks
rcParams["xtick.minor.visible"] = True
rcParams["ytick.minor.visible"] = True

# Slightly thicker axes and lines for retina displays
rcParams["axes.linewidth"] = 1.2
rcParams["lines.linewidth"] = 1.5

# Legend
rcParams["legend.framealpha"] = 0.9
rcParams["legend.fontsize"] = 12

rcParams["image.cmap"] = _cmap.name
rcParams["axes.prop_cycle"] = cycler(color=_cmap.colors)

In [ ]:
import os
import sys
import time

sys.path.append("/home/ssage/Karmma/Galaxy_KaRMMa/")

import h5py as h5
import healpy as hp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from blackjax.diagnostics import effective_sample_size

jax.config.update("jax_enable_x64", True)

from karmma import KarmmaConfig, KarmmaSampler
from karmma.structs import KarmmaPosition, ThetaParams, XlmParams

## User Inputs

Edit the values below, then run the rest of the notebook top to bottom.

In [ ]:
# ============================================================
# USER INPUTS — edit these for each analysis
# ============================================================

# Shared analysis config (mask, C_ell, alpha, beta, N_bar, pixwin, etc. — same for every run)
configfile = "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Inputs/desy3.yaml"

# One Outputs_* directory per run — each must contain samples.h5 and mcmc_metadata.h5
output_dirs = [
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Outputs_100",
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Outputs_175",
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Outputs_275",
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Outputs_400",
]

# One mock_dg_*.h5 path per run, matching output_dirs by position
mock_dg_paths = [
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Inputs/mock_dg_0.h5",
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Inputs/mock_dg_0.h5",
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Inputs/mock_dg_0.h5",
    "/spiff/ssage/karmma_data/LN_dg_RS_linmu_bias/Inputs/mock_dg_0.h5",
]

# Optional custom labels, one per run; leave as None to auto-generate "Run 0", "Run 1", ...
run_labels = [
    "100",
    "175",
    "275",
    "400",
]

# Path to theta_true.h5 (shared across all runs); leave as None to use <io_dir>/Inputs/theta_true.h5
theta_true_path = None

assert len(output_dirs) == len(mock_dg_paths), (
    f"output_dirs ({len(output_dirs)}) and mock_dg_paths ({len(mock_dg_paths)}) "
    "must have the same length"
)
n_runs = len(output_dirs)

if run_labels is None:
    run_labels = [f"Run {i}" for i in range(n_runs)]
assert len(run_labels) == n_runs, "run_labels must match the number of runs"

# One consistent color per run, reused across every plot below
_palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
run_colors = [_palette[i % len(_palette)] for i in range(n_runs)]

# Width for the "Run" label column in printed tables, so labels of any length line up
label_w = max(len(label) for label in run_labels) + 2

## Load Configuration

In [ ]:
config = KarmmaConfig(configfile)

analysis = config.analysis
io = config.io
mcmc = config.mcmc

nside = analysis.nside
nbins = analysis.nbins
io_dir = io.io_dir
n_samples = mcmc.n_samples

if theta_true_path is None:
    theta_true_path = os.path.join(io_dir, "Inputs/theta_true.h5")

## Truth Values & Per-Run Log-Probability

In [ ]:
# theta_true is identical across all runs
with h5.File(theta_true_path, "r") as f:
    theta_true_arr = np.stack(
        [f[f"theta/{field}"][:] for field in ThetaParams._fields], axis=-1
    )
    # shape: (nbins, 6)
theta_true_pos = ThetaParams(*[jnp.array(theta_true_arr[:, j]) for j in range(6)])

# Build one sampler per run; xlm_true and dg_obs differ per run
log_prob_jits = []
lp_truths = []
for i, mock_path in enumerate(mock_dg_paths):
    with h5.File(mock_path, "r") as f:
        dg_obs_i = f["dg_obs"][:]
        xlm_true_i = XlmParams(
            real=jnp.array(f["true_xlm/real"][:]), imag=jnp.array(f["true_xlm/imag"][:])
        )
    s = KarmmaSampler(
        dg_obs=dg_obs_i,
        mask=io.mask,
        CL=analysis.cl,
        alpha=analysis.alpha,
        beta=analysis.beta,
        N_bar=io.N_bar,
        infer_theta=True,
        pixwin=analysis.pixwin,
    )
    lp_jit = jax.jit(s.log_prob)
    lp_truth_i = float(lp_jit(KarmmaPosition(xlm=xlm_true_i, theta=theta_true_pos)))
    log_prob_jits.append(lp_jit)
    lp_truths.append(lp_truth_i)
    print(f"{run_labels[i]}: log_prob at truth = {lp_truth_i:.2f}")

## MCMC Diagnostics

### Summary Table

In [ ]:
runs = []
for d in output_dirs:
    with h5.File(os.path.join(d, "mcmc_metadata.h5"), "r") as f:
        runs.append(
            {
                "seed": f["seed"][()],
                "step_size": f["step_size"][()],
                "acceptance_rate": f["acceptance_rate"][:],
                "is_divergent": f["is_divergent"][:],
                "num_integration_steps": f["num_integration_steps"][:],
                "energy": f["energy"][:],
                "warmup_acceptance_rate": f["warmup_acceptance_rate"][:],
                "warmup_is_divergent": f["warmup_is_divergent"][:],
                "warmup_num_integration_steps": f["warmup_num_integration_steps"][:],
                "inv_mass_matrix": f["inverse_mass_matrix"][:],
            }
        )

col_w = 12
header = (
    f"{'Run':<{label_w}}{'Seed':>{col_w}}{'Step size':>{col_w}}"
    f"{'Divs(samp)':>{col_w}}{'Divs(warm)':>{col_w}}"
    f"{'Acc(samp)':>{col_w}}{'Acc(warm)':>{col_w}}"
)
print(header)
print("-" * len(header))
for label, r in zip(run_labels, runs, strict=False):
    print(
        f"{label:<{label_w}}{r['seed']:>{col_w}}{r['step_size']:>{col_w}.4f}"
        f"{int(r['is_divergent'].sum()):>{col_w}}{int(r['warmup_is_divergent'].sum()):>{col_w}}"
        f"{r['acceptance_rate'].mean():>{col_w}.4f}{r['warmup_acceptance_rate'].mean():>{col_w}.4f}"
    )

### Acceptance Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, (r, label) in enumerate(zip(runs, run_labels, strict=False)):
    ax.plot(r["acceptance_rate"], color=run_colors[i], linewidth=0.8, alpha=0.8, label=label)
    ax.axhline(
        r["acceptance_rate"].mean(),
        color=run_colors[i],
        linestyle="--",
        linewidth=1.0,
        alpha=0.6,
    )
ax.axhline(
    mcmc.target_acceptance,
    color="k",
    linestyle="--",
    linewidth=1.0,
    label=f"Target: {mcmc.target_acceptance:.2f}",
)
ax.set_xlabel("Sample")
ax.set_ylabel("Acceptance rate")
ax.set_xlim(0, n_samples)
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

### Integration Steps

In [ ]:
all_unique = np.unique(np.concatenate([r["num_integration_steps"] for r in runs]))
x = np.arange(len(all_unique))
bar_width = 0.8 / n_runs

fig, ax = plt.subplots(figsize=(8, 4))
for i, (r, label) in enumerate(zip(runs, run_labels, strict=False)):
    counts = np.array([(r["num_integration_steps"] == s).sum() for s in all_unique])
    offset = (i - (n_runs - 1) / 2) * bar_width
    ax.bar(
        x + offset,
        counts,
        bar_width,
        label=label,
        color=run_colors[i],
        edgecolor="k",
        linewidth=0.5,
    )
ax.set_xticks(x)
ax.set_xticklabels(all_unique)
ax.set_xlabel("Number of integration steps")
ax.set_ylabel("Counts")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

## Posterior Samples & Effective Sample Size

In [ ]:
all_xlm_real = []
all_xlm_imag = []
all_theta_samples = []

for d in output_dirs:
    with h5.File(os.path.join(d, "samples.h5"), "r") as f:
        all_xlm_real.append(f["xlm/real"][:])
        all_xlm_imag.append(f["xlm/imag"][:])
        all_theta_samples.append(
            np.stack([f[f"theta/{field}"][:] for field in ThetaParams._fields], axis=-1)
        )  # (n_samples, nbins, 6)

all_ess_real = [effective_sample_size(r[np.newaxis]) for r in all_xlm_real]
all_ess_imag = [effective_sample_size(r[np.newaxis]) for r in all_xlm_imag]

for label, er, ei in zip(run_labels, all_ess_real, all_ess_imag, strict=False):
    print(
        f"{label}:  xlm_real — min={er.min():.1f}, median={np.median(er):.1f}, mean={er.mean():.1f}"
        f"  |  xlm_imag — min={ei.min():.1f}, median={np.median(ei):.1f}, mean={ei.mean():.1f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, (label, er, ei) in enumerate(zip(run_labels, all_ess_real, all_ess_imag, strict=False)):
    axes[0].hist(
        er.flatten(), bins=30, color=run_colors[i], alpha=0.5, edgecolor="none", label=label
    )
    axes[0].axvline(er.min(), color=run_colors[i], linestyle="--", linewidth=1.0)
    axes[1].hist(
        ei.flatten(), bins=30, color=run_colors[i], alpha=0.5, edgecolor="none", label=label
    )
    axes[1].axvline(ei.min(), color=run_colors[i], linestyle="--", linewidth=1.0)

axes[0].set_xlabel("ESS")
axes[0].set_ylabel("Counts")
axes[0].set_title(r"$x_{\ell m}$ real")
axes[0].legend()
axes[1].set_xlabel("ESS")
axes[1].set_ylabel("Counts")
axes[1].set_title(r"$x_{\ell m}$ imag")
axes[1].legend()
plt.tight_layout()
plt.show()

### Theta ESS

In [ ]:
if mcmc.infer_theta:
    theta_param_names_plain = ["A_t", "log_T", "c", "log_R", "mu0", "a"]
    theta_param_names_latex = [
        r"$A_t$",
        r"$\log T$",
        r"$c$",
        r"$\log R$",
        r"$\mu_0$",
        r"$a$",
    ]

    all_ess_theta = [
        np.array(effective_sample_size(ts[np.newaxis]))  # (nbins, 6)
        for ts in all_theta_samples
    ]

    for label, ess_t in zip(run_labels, all_ess_theta, strict=False):
        print(
            f"{label}: theta — min={ess_t.min():.1f}, median={np.median(ess_t):.1f}, mean={ess_t.mean():.1f}"
        )

    x = np.arange(len(theta_param_names_latex))
    fig, axes = plt.subplots(1, nbins, figsize=(7 * nbins, 4), sharey=True)
    if nbins == 1:
        axes = [axes]

    for b, ax in enumerate(axes):
        for i, (label, ess_t) in enumerate(zip(run_labels, all_ess_theta, strict=False)):
            ax.scatter(x, ess_t[b], color=run_colors[i], label=label, zorder=3, s=50)
        ax.axhline(
            n_samples,
            color="k",
            linestyle="--",
            linewidth=1.0,
            label=f"$n_{{\\rm samples}}={n_samples}$",
        )
        ax.set_xticks(x)
        ax.set_xticklabels(theta_param_names_latex)
        ax.set_xlabel(r"$\theta$ parameter")
        ax.set_ylabel("ESS")
        ax.set_title(f"Bin {b + 1}")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, bbox_to_anchor=(1.02, 0.5), loc="center left", borderaxespad=0
    )
    plt.suptitle(r"ESS per $\theta$ parameter")
    plt.tight_layout()
    plt.show()

### Theta Traces

In [ ]:
if mcmc.infer_theta:
    n_params = len(theta_param_names_plain)
    fig, axes = plt.subplots(
        n_params, nbins, figsize=(7 * nbins, 2.2 * n_params), sharex=True, squeeze=False
    )

    for b in range(nbins):
        for row in range(n_params):
            ax = axes[row, b]
            for i, (ts, label) in enumerate(zip(all_theta_samples, run_labels, strict=False)):
                ax.plot(
                    ts[:, b, row], color=run_colors[i], linewidth=0.7, alpha=0.6, label=label
                )
            ax.axhline(
                theta_true_arr[b, row],
                color="k",
                linestyle="--",
                linewidth=1.0,
                label=f"Truth: {theta_true_arr[b, row]:.3f}",
            )
            ax.set_ylabel(theta_param_names_latex[row])
            ax.set_title(rf"Bin {b + 1}, {theta_param_names_latex[row]}", fontsize=12)

    for b in range(nbins):
        axes[-1, b].set_xlabel("Sample")
        axes[-1, b].set_xlim(0, n_samples)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, bbox_to_anchor=(1.02, 0.5), loc="center left", borderaxespad=0
    )
    plt.suptitle(r"$\theta$ traces", y=1.01)
    plt.tight_layout()
    plt.show()

## Mass Matrix Conditioning

In [ ]:
gen_lmax = 3 * nside - 1
gen_ell, gen_emm = hp.Alm.getlm(gen_lmax)
real_pos = np.where(gen_ell > 1)[0]
imag_pos = np.where((gen_ell > 1) & (gen_emm > 0))[0]
n_real = len(real_pos)
n_imag = len(imag_pos)

# inv_mass_matrix layout (flat, from ravel_pytree(KarmmaPosition)):
# [xlm.real (nbins*n_real), xlm.imag (nbins*n_imag), theta fields in ThetaParams order (nbins each)]
theta_names = ThetaParams._fields
n_theta = len(theta_names) * nbins

col_w = 12
for kind, n_kind, offset in [("real", n_real, 0), ("imag", n_imag, nbins * n_real)]:
    header = (
        f"{'Run':<{label_w}}{'Min':>{col_w}}{'Max':>{col_w}}{'Mean':>{col_w}}"
        f"{'Std':>{col_w}}{'Cond':>{col_w}}{'Inf/0':>{col_w}}"
    )
    print(f"xlm ({kind}) inverse mass matrix")
    print(header)
    print("-" * len(header))
    for label, r in zip(run_labels, runs, strict=False):
        block = r["inv_mass_matrix"][offset : offset + nbins * n_kind].reshape(nbins, n_kind)
        bad = np.any(block == 0) or np.any(~np.isfinite(block))
        print(
            f"{label:<{label_w}}{block.min():>{col_w}.2e}{block.max():>{col_w}.2e}"
            f"{block.mean():>{col_w}.2e}{block.std():>{col_w}.2e}"
            f"{block.max() / block.min():>{col_w}.2e}{str(bad):>{col_w}}"
        )
    print()

if mcmc.infer_theta:
    header = (
        f"{'Run':<{label_w}}"
        + "".join(f"{name:>{col_w}}" for name in theta_names)
        + f"{'Inf/0':>{col_w}}"
    )
    print("theta inverse mass matrix (mean over bins, by parameter)")
    print(header)
    print("-" * len(header))
    for label, r in zip(run_labels, runs, strict=False):
        theta_block = r["inv_mass_matrix"][-n_theta:].reshape(len(theta_names), nbins)
        bad = np.any(theta_block == 0) or np.any(~np.isfinite(theta_block))
        row = "".join(f"{theta_block[k].mean():>{col_w}.2e}" for k in range(len(theta_names)))
        print(f"{label:<{label_w}}{row}{str(bad):>{col_w}}")

## Theta Corner Plots

In [ ]:
if mcmc.infer_theta:
    from getdist import MCSamples, plots

    theta_labels_getdist = ["A_t", "log(T)", "c", "log(R)", "mu_0", "a"]
    theta_init = np.array(io.initial_position.theta).T  # (nbins, 6)
    line_args = [{"color": c, "lw": 1.5} for c in run_colors]

    for b in range(nbins):
        mc_list = [
            MCSamples(
                samples=ts[:, b],
                names=theta_param_names_plain,
                labels=theta_labels_getdist,
                label=label,
            )
            for ts, label in zip(all_theta_samples, run_labels, strict=False)
        ]
        g = plots.get_subplot_plotter()
        g.triangle_plot(
            mc_list,
            filled=True,
            colors=run_colors,
            line_args=line_args,
            markers={theta_param_names_plain[j]: theta_init[b, j] for j in range(6)},
        )
        g.fig.suptitle(rf"Bin {b + 1} $\theta$ posteriors", y=1.01)
        plt.show()

## Log-Probability Consistency Check

In [ ]:
all_lp = []
for run_i, (label, d, xlm_r, xlm_i, ts) in enumerate(
    zip(run_labels, output_dirs, all_xlm_real, all_xlm_imag, all_theta_samples, strict=False)
):
    meta_path = os.path.join(d, "mcmc_metadata.h5")
    with h5.File(meta_path, "r") as f:
        if "log_prob" in f:
            all_lp.append(np.array(f["log_prob"]))
            print(f"{label}: loaded log_prob from metadata")
            continue

    print(f"{label}: log_prob not in metadata, recomputing...")
    lp_jit = log_prob_jits[run_i]
    n_samp = xlm_r.shape[0]
    lp_run = np.zeros(n_samp)
    t0 = time.perf_counter()
    for j in range(n_samp):
        pos = KarmmaPosition(
            xlm=XlmParams(real=jnp.array(xlm_r[j]), imag=jnp.array(xlm_i[j])),
            theta=ThetaParams(*[jnp.array(ts[j, :, k]) for k in range(6)]),
        )
        lp_run[j] = float(lp_jit(pos))
        if (j + 1) % 100 == 0:
            print(f"  {j + 1}/{n_samp}  ({time.perf_counter() - t0:.1f}s)")
    all_lp.append(lp_run)

    with h5.File(meta_path, "a") as f:
        f["log_prob"] = lp_run
    print(f"{label}: saved log_prob to metadata")

fig, ax = plt.subplots(figsize=(10, 4))
for i, (label, lp_run) in enumerate(zip(run_labels, all_lp, strict=False)):
    diffs = lp_run - lp_truths[i]
    ax.plot(diffs, color=run_colors[i], linewidth=0.7, alpha=0.8, label=label)
    ax.axhline(diffs.mean(), color=run_colors[i], linestyle="--", linewidth=1.0, alpha=0.6)
ax.axhline(0, color="k", linestyle="--", linewidth=1.0, label="Truth")
ax.set_xlabel("Sample")
ax.set_ylabel(r"$\Delta\log P$ (sample $-$ truth)")
ax.set_xlim(0, n_samples)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()